# 04 Lazy Evaluation

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand <b>why</b> Spark is lazy, prove it with <code>explain()</code> and timing, see the optimizations laziness enables (filter pushdown, column pruning, pipelining, short-circuiting with <code>limit</code>), and know its costs: recomputation on every action and errors that appear late.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Lazy evaluation is the idea that makes Spark's optimizer possible. Because Spark sees your <b>whole</b> query before running it, it can reorder and combine your steps. Interviewers ask "why is Spark lazy?" and expect more than "it waits for an action": they want the benefits and the trade-offs.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Why wait?</b><br>
When you call transformations, Spark only builds a <b>logical plan</b>. At the first action, it optimizes the <b>entire</b> plan and then executes it. That allows:
<ul><li><b>Predicate pushdown</b>: filters move as early as possible, even into the file scan</li><li><b>Column pruning</b>: only the columns that are used are read</li><li><b>Pipelining</b>: consecutive narrow steps run together in one pass over the data (whole-stage code generation)</li><li><b>Short-circuiting</b>: <code>limit(10)</code> or <code>first()</code> can stop after a few rows</li><li><b>Skipping unused work</b>: transformations that never reach an action never run</li></ul>
The trade-offs:
<ul><li>Each action <b>recomputes</b> the plan from the source, unless the result is materialized</li><li>Errors are raised at the action, not at the line that caused them</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An analyst writes: read 3 years of clickstream data (60 columns), add 10 derived columns, then filter to yesterday and keep 4 columns. Executed eagerly, step by step, this would process 3 years × 70 columns. Because Spark is lazy, the optimizer pushes the date filter into the scan and reads only the 4 columns needed, so the query touches one day of 4 columns.
</div>

## 1. Proving laziness with timing

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines an expensive pipeline and times the definition, then times the action.<br><br>
<b>Why it matters</b><br>The definition is instant no matter how much data is involved. The work happens only at the action.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Definition in a few milliseconds, then the action takes much longer.
</div>

In [0]:
import time
from pyspark.sql import functions as F

start = time.time()
pipeline = (
    spark.range(10_000_000)
    .withColumn("a", F.sqrt("id"))
    .withColumn("b", F.col("a") * 2)
    .filter(F.col("b") > 100)
)
print(f"definition: {time.time() - start:.4f}s")

start = time.time()
pipeline.agg(F.sum("b")).first()
print(f"action    : {time.time() - start:.4f}s")

## 2. The optimizer rewrites your query: filter pushdown

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a filter <b>after</b> a derived column and a projection, then shows that the optimized plan applies the filter first.<br><br>
<b>Why it matters</b><br>You don't have to hand-order your transformations for performance. The optimizer moves the filter down because it sees the whole plan.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
In the <b>Analyzed</b> plan, the <code>Filter</code> is at the top (where you wrote it). In the <b>Optimized</b> plan, the filter on <code>id</code> is pushed below the projection, right above the <code>Range</code> source.
</div>

In [0]:
written_late = (
    spark.range(1_000_000)
    .withColumn("double", F.col("id") * 2)
    .select("id", "double")
    .filter(F.col("id") < 100)          # written last
)
written_late.explain("extended")

## 3. Column pruning

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a wide DataFrame, writes it to a Parquet folder, and reads it back selecting only 2 columns.<br><br>
<b>Why it matters</b><br>Columns you never use are never read. The <code>ReadSchema</code> in the scan lists only the columns the query needs, even though the code read "the whole table".
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A <code>FileScan parquet</code> whose <code>ReadSchema</code> contains only <code>id</code> and <code>c3</code>.
</div>

In [0]:
path = "/Volumes/workspace/spark_basics/raw_files/lesson02_04/wide"
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spark_basics")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.spark_basics.raw_files")

In [0]:
wide = spark.range(100_000).select("id", *[(F.col("id") * i).alias(f"c{i}") for i in range(20)])
wide.write.mode("overwrite").parquet(path)

spark.read.parquet(path).select("id", "c3").filter("c3 > 10").explain()

## 4. Short-circuiting with `limit`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Compares the time to get 5 rows with <code>limit(5)</code> and the time to fully process the same DataFrame.<br><br>
<b>Why it matters</b><br>Because Spark knows only 5 rows are needed, it can stop early instead of processing everything. An eager system would compute the full result first.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>limit(5).collect()</code> returns almost immediately, while the full aggregation over 100 million rows takes much longer.
</div>

In [0]:
big = spark.range(100_000_000).withColumn("v", F.sqrt(F.col("id")) * F.log1p("id"))

start = time.time()
big.limit(5).collect()
print(f"limit(5)          : {time.time() - start:.3f}s")

start = time.time()
big.agg(F.sum("v")).collect()
print(f"full aggregation  : {time.time() - start:.3f}s")

## 5. The cost: every action recomputes

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds a column with a random <code>uuid()</code> and collects twice. Different values prove that the plan ran again.<br><br>
<b>Why it matters</b><br>A DataFrame is a <b>recipe</b>, not a stored result. Each action cooks it again. With non-deterministic functions such as <code>uuid()</code>, two actions can even give <b>different</b> answers. (<code>rand()</code> is a special case: it gets a seed when the column is created, so the same DataFrame repeats its values, but rebuilding the DataFrame gives new ones.) Materialize results you reuse by writing a table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>same values: False</code>: the two collects returned different UUIDs, because the plan was executed twice.
</div>

In [0]:
recipe = spark.range(3).withColumn("token", F.expr("uuid()"))

first_run = [r["token"] for r in recipe.collect()]
second_run = [r["token"] for r in recipe.collect()]
print(first_run)
print(second_run)
print("same values:", first_run == second_run)

## 6. Materialize to stop recomputation

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the DataFrame once to a table, then reads the table twice.<br><br>
<b>Why it matters</b><br>Writing an intermediate result (a Delta table on Databricks) freezes it: later actions read the stored data instead of recomputing. On serverless, this replaces <code>cache()</code> for expensive, reused results.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>same values: True</code> after materializing.
</div>

In [0]:
table = "workspace.spark_basics.lesson02_04_tokens"

In [0]:
recipe.write.mode("overwrite").saveAsTable(table)
stored = spark.table(table)
a = sorted(r["token"] for r in stored.collect())
b = sorted(r["token"] for r in stored.collect())
print("same values:", a == b)
spark.sql(f"DROP TABLE IF EXISTS {table}")

## Under the hood

```
transformation calls      ->  logical plan grows (unresolved)
first action              ->  Analyzer: resolve names, types
                              Optimizer: pushdown, pruning, constant folding, collapsing projects ...
                              Planner:   choose physical operators (joins, aggregates)
                              Codegen:   fuse narrow operators into one function per stage
                              Execute
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> lazy: just plan nodes.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> triggers the whole pipeline above.

**Pipelining:** in the plan below, `Range`, `Project` and `Filter` share one `*(1)` codegen stage. Spark doesn't create an intermediate dataset for each step: each row flows through all three operators in one loop. That's only possible because Spark saw all the steps before running any of them.

In [0]:
pipeline.explain()

In [0]:
# Optional cleanup
dbutils.fs.rm("/Volumes/workspace/spark_basics/raw_files/lesson02_04", True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: results change between two actions</b><br>
Non-deterministic functions such as <code>uuid()</code> are recomputed per action. Materialize the result, and seed <code>rand(seed)</code> so rebuilding the DataFrame gives the same values.<br><br>
<b>⛔ Problem: an expensive DataFrame is used by several actions and the job is slow</b><br>Each action recomputes it. Write it to a table once and read from there.<br><br>
<b>⛔ Problem: timing a cell that only defines transformations</b><br>It measures nothing. Time the action.<br><br>
**⛔ Problem: an error at <code>write</code> comes from an early transformation**<br>Lazy evaluation delays data errors to the action. Test transformations on a small sample with <code>limit</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is lazy evaluation in Spark?</b><br>
Transformations aren't executed when they're called. Spark records them in a logical plan, and only when an action is called does it optimize the whole plan and execute it.<br><br>

<b>🎤 2. Why is Spark lazy? What are the benefits?</b><br>
Seeing the whole query lets the optimizer push filters down, prune unused columns, combine consecutive operations into one pass, choose join strategies, and stop early for <code>limit</code>. It also avoids computing anything that is never used.<br><br>

<b>🎤 3. What are the downsides of lazy evaluation?</b><br>
Each action recomputes the plan from the source unless the result is materialized, and errors appear at the action rather than where they were caused, which makes debugging harder.<br><br>

<b>🎤 4. How can you prove Spark is lazy?</b><br>
Time the definition of transformations (instant) against the action, or create a DataFrame whose execution would fail, such as dividing by zero, and show that defining it and printing its schema work while the action fails.<br><br>

<b>🎤 5. Can two actions on the same DataFrame return different results?</b><br>
Yes, if the plan contains non-deterministic functions such as <code>uuid()</code>, because each action recomputes them. Materialize the result to make it stable. <code>rand()</code> gets its seed when the expression is created, so it repeats within the same DataFrame but changes when the DataFrame is rebuilt without a seed.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook cell defines 15 transformations on a 2 TB table and finishes in under a second. What explains this?</b><br>
A. The data is cached by default<br>
B. Transformations are lazy, so nothing was read or computed yet<br>
C. Photon processed the data instantly<br>
D. Delta Lake precomputes all transformations<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A query selects 3 columns from a Delta table and filters on one of them, but the code reads the table first and filters at the end. What does Spark read from storage?</b><br>
A. All columns of all files, then filters<br>
B. Only the 3 columns, and it can skip files using the filter<br>
C. Only the filtered column<br>
D. It depends on the order of the code<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Column pruning and predicate pushdown apply to the whole optimized plan, whatever order the code was written in.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a query where a filter appears after a <code>join</code>, and show in <code>explain(True)</code> that the optimizer pushed it below the join</li><li>Show that <code>F.rand()</code> without a seed repeats its values on two actions of the <b>same</b> DataFrame, but changes when you <b>rebuild</b> the DataFrame, while <code>F.rand(seed=1)</code> is stable in both cases</li><li>Read <code>path</code> and select only <code>c7</code>. Confirm with <code>explain</code> that only <code>c7</code> is read</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Filter written after the join is pushed into the left side
left = spark.range(100_000).withColumn("k", F.col("id") % 100)
right = spark.range(100).withColumnRenamed("id", "k")
left.join(right, "k").filter(F.col("id") < 50).explain(True)

# 2. rand() picks its seed when the column is created
total = lambda df: df.agg(F.sum("r")).first()[0]
unseeded = spark.range(1000).withColumn("r", F.rand())
print("unseeded, same DataFrame twice:", total(unseeded) == total(unseeded))
print("unseeded, rebuilt DataFrame   :", total(unseeded) == total(spark.range(1000).withColumn("r", F.rand())))
seeded = lambda: spark.range(1000).withColumn("r", F.rand(seed=1))
print("seeded, rebuilt DataFrame     :", total(seeded()) == total(seeded()))

# 3. Only c7 in ReadSchema
spark.read.parquet(path).select("c7").explain()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Transformations build a plan. Actions optimize and execute the whole plan</li><li>Benefits: pushdown, pruning, pipelining (codegen), early stop with <code>limit</code>, skipping unused work</li><li>Costs: recomputation on every action, and late errors</li><li>Non-deterministic functions such as <code>uuid()</code> can differ between actions. Materialize, and seed <code>rand</code></li><li>On serverless, materialize reused results by writing a table (no <code>cache</code>)</li><li>Prove laziness with timing, or with a DataFrame that fails only at the action</li></ul>
</div>

| Check | Code |
|---|---|
| See all plan phases | `df.explain("extended")` or `df.explain(True)` |
| Pushed filters and pruned columns | `explain()` → `PushedFilters`, `ReadSchema` |
| Stop early | `df.limit(n)` |
| Stable random | `F.rand(seed=42)` |
| Freeze a result | `df.write.saveAsTable("t")` |

## Git commit

```
git add 02_spark_internals/04_lazy_evaluation.ipynb
git commit -m "add 02_04 lazy evaluation notebook"
```